In [ ]:
# !pip install yfinance
# !pip install TA-Lib 
# !pip install numpy
# !pip install pandas
# !pip install vectorbt
# !pip install scipy

In [ ]:
import sys, os
# Ensure lib/ is importable (for Google Sheets logging)
for _p in ['/content/trading-strategies', os.path.join(os.getcwd(), '..')]:
    if os.path.isdir(os.path.join(_p, 'lib')) and _p not in sys.path:
        sys.path.insert(0, _p)

import yfinance as yf
import talib
import numpy as np
import pandas as pd
import vectorbt as vbt
import warnings
from scipy import stats
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
pd.set_option('future.no_silent_downcasting', True)

In [ ]:
# DOWNLOAD STOCK DATA FROM 2018 USING YFINANCE

# Configuration - Change these variables as needed
TICKER = 'SPY'  # Best on indices (SPY, QQQ, ^GDAXI, ^DJI)
START_DATE = '2018-01-01'

stock_data = yf.download(TICKER, start=START_DATE, interval='1d')

if not stock_data.empty:
    print(f'Successfully downloaded {len(stock_data)} records for {TICKER} from {START_DATE}')
    print(f'Data range: {stock_data.index.min().date()} to {stock_data.index.max().date()}')
    print('\nFirst 5 rows:')
    print(stock_data.head())
else:
    print(f'Failed to download {TICKER} data from yfinance')

In [ ]:
# DAY-OF-WEEK ANALYSIS

# Handle MultiIndex columns from yfinance
if isinstance(stock_data.columns, pd.MultiIndex):
    close = stock_data[('Close', TICKER)].astype(float)
    high = stock_data[('High', TICKER)].astype(float)
    low = stock_data[('Low', TICKER)].astype(float)
    open_ = stock_data[('Open', TICKER)].astype(float)
    volume = stock_data[('Volume', TICKER)].astype(float)
else:
    close = stock_data['Close'].astype(float)
    high = stock_data['High'].astype(float)
    low = stock_data['Low'].astype(float)
    open_ = stock_data['Open'].astype(float)
    volume = stock_data['Volume'].astype(float)

# Day-of-week distribution
dow = close.index.dayofweek  # 0=Mon, 4=Fri
dow_names = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']
daily_returns = close.pct_change()

print('Trading Day Distribution:')
for d in range(5):
    n = (dow == d).sum()
    avg_ret = daily_returns[dow == d].mean() * 100
    print(f'  {dow_names[d]:12s}: {n:5d} bars, avg return: {avg_ret:+.4f}%')

# Quick bar chart of avg returns by day
fig, ax = plt.subplots(1, 1, figsize=(8, 4))
avg_by_day = [daily_returns[dow == d].mean() * 100 for d in range(5)]
colors = ['green' if r > 0 else 'red' for r in avg_by_day]
ax.bar(dow_names, avg_by_day, color=colors, alpha=0.7)
ax.set_ylabel('Average Daily Return (%)')
ax.set_title(f'{TICKER} - Average Return by Day of Week')
ax.axhline(y=0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# PREPARE PRICE SERIES + TRAIN / VAL SPLIT

TRAIN_RATIO = 0.60
split_idx = int(len(close) * TRAIN_RATIO)

train_close = close.iloc[:split_idx]
train_high = high.iloc[:split_idx]
train_low = low.iloc[:split_idx]

val_close = close.iloc[split_idx:]
val_high = high.iloc[split_idx:]
val_low = low.iloc[split_idx:]

print(f'Train: {train_close.index[0].date()} to {train_close.index[-1].date()} ({len(train_close)} bars)')
print(f'Val:   {val_close.index[0].date()} to {val_close.index[-1].date()} ({len(val_close)} bars)')
print(f'Split ratio: {TRAIN_RATIO:.0%} / {1-TRAIN_RATIO:.0%}')

## Tom Hougaard - Scenario Strategy

A pure price-action weekly pattern strategy. No indicators -- just OHLC and day-of-week.

### Scenario A (Mid-week)
- **Short:** Monday HIGH > Tuesday HIGH AND Monday HIGH > Wednesday HIGH --> short Wednesday close, exit after N days
- **Long (mirror):** Monday LOW < Tuesday LOW AND Monday LOW < Wednesday LOW --> long Wednesday close

### Scenario B (End-of-week)
- **Short:** Thursday HIGH > Friday HIGH --> short Friday close, exit after N days
- **Long (mirror):** Thursday LOW < Friday LOW --> long Friday close

### Grid Search
- `scenario`: A_short, A_long, B_short, B_long, combined
- `holding_days`: 1, 2, 3
- `require_consec`: require consecutive declining highs/lows (stricter filter)
- `min_range_pct`: minimum Monday range to filter low-volatility weeks

In [ ]:
# DEFINE PARAMETER RANGES

scenarios = ['A_short', 'A_long', 'B_short', 'B_long', 'A_short+B_short', 'A_long+B_long']
holding_days_range = [1, 2, 3]
require_consec_range = [True, False]
min_range_pct_range = [0.0, 0.005, 0.01]

total_combos = len(scenarios) * len(holding_days_range) * len(require_consec_range) * len(min_range_pct_range)

print('Parameter Ranges:')
print(f'  scenarios:       {scenarios}')
print(f'  holding_days:    {holding_days_range}')
print(f'  require_consec:  {require_consec_range}')
print(f'  min_range_pct:   {min_range_pct_range}')
print(f'\nTotal combinations: {total_combos}')

In [ ]:
# SIGNAL GENERATION + GRID SEARCH

INIT_CASH = 100_000
FEES = 0.0005
SLIPPAGE = 0.0005
MIN_TRADES = 10

def build_week_map(price_index):
    """Build a mapping from each trading day to its week's Mon-Fri indices.
    Returns a DataFrame with columns for each weekday's index position."""
    dow = price_index.dayofweek
    # Group by ISO calendar week
    week_labels = pd.Series(
        [f'{d.isocalendar()[0]}-{d.isocalendar()[1]:02d}' for d in price_index],
        index=price_index
    )
    return week_labels


def generate_scenario_signals(close_s, high_s, low_s, scenario, holding_days=1,
                               require_consec=False, min_range_pct=0.0):
    """
    Generate entry/exit signals for Hougaard Scenario Strategy.
    
    Returns: entries (Series[bool]), exits (Series[bool]), direction ('longonly'/'shortonly')
    """
    idx = close_s.index
    dow = idx.dayofweek  # 0=Mon, 4=Fri
    entries = pd.Series(False, index=idx)
    exits = pd.Series(False, index=idx)
    
    # Determine component scenarios
    if '+' in scenario:
        parts = scenario.split('+')
        direction = 'shortonly' if 'short' in parts[0] else 'longonly'
    else:
        parts = [scenario]
        direction = 'shortonly' if 'short' in scenario else 'longonly'
    
    # Build arrays for fast lookup: for each bar, get prev days' highs/lows
    h = high_s.values
    l = low_s.values
    c = close_s.values
    dw = dow.values if hasattr(dow, 'values') else np.array(dow)
    n = len(c)
    
    # Precompute: for each bar i, find the positions of this week's days
    # We iterate and track week boundaries
    entry_flags = np.zeros(n, dtype=bool)
    
    for part in parts:
        is_short = 'short' in part
        is_A = part.startswith('A')
        is_B = part.startswith('B')
        
        for i in range(4, n):  # need at least a few lookback days
            if is_A and dw[i] == 2:  # Wednesday
                # Find this week's Monday and Tuesday
                # Walk back to find Mon (dw=0) and Tue (dw=1)
                mon_idx = tue_idx = None
                for j in range(i-1, max(i-6, -1), -1):
                    if dw[j] == 1 and tue_idx is None:
                        tue_idx = j
                    if dw[j] == 0 and mon_idx is None:
                        mon_idx = j
                if mon_idx is None or tue_idx is None:
                    continue
                
                # Min range filter (Monday's range as pct of close)
                if min_range_pct > 0:
                    mon_range = (h[mon_idx] - l[mon_idx]) / c[mon_idx]
                    if mon_range < min_range_pct:
                        continue
                
                if is_short:
                    # Mon HIGH > Tue HIGH AND Mon HIGH > Wed HIGH
                    cond = h[mon_idx] > h[tue_idx] and h[mon_idx] > h[i]
                    if require_consec:
                        cond = cond and h[tue_idx] > h[i]  # consecutive decline
                else:
                    # Mon LOW < Tue LOW AND Mon LOW < Wed LOW
                    cond = l[mon_idx] < l[tue_idx] and l[mon_idx] < l[i]
                    if require_consec:
                        cond = cond and l[tue_idx] < l[i]
                
                if cond:
                    entry_flags[i] = True
            
            elif is_B and dw[i] == 4:  # Friday
                # Find this week's Thursday
                thu_idx = None
                for j in range(i-1, max(i-4, -1), -1):
                    if dw[j] == 3:
                        thu_idx = j
                        break
                if thu_idx is None:
                    continue
                
                if is_short:
                    cond = h[thu_idx] > h[i]
                else:
                    cond = l[thu_idx] < l[i]
                
                if cond:
                    entry_flags[i] = True
    
    # Generate exits: holding_days after entry
    exit_flags = np.zeros(n, dtype=bool)
    for i in range(n):
        if entry_flags[i]:
            exit_idx = i + holding_days
            if exit_idx < n:
                exit_flags[exit_idx] = True
    
    # Apply 1-bar execution delay
    entries = pd.Series(entry_flags, index=idx).shift(1).fillna(False).astype(bool)
    exits = pd.Series(exit_flags, index=idx).shift(1).fillna(False).astype(bool)
    
    return entries, exits, direction


def run_backtest(entries, exits, price, direction='longonly'):
    """Run vectorbt backtest."""
    pf = vbt.Portfolio.from_signals(
        price, entries=entries, exits=exits,
        direction=direction,
        init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq='D'
    )
    return pf


def get_metrics(pf):
    """Extract metrics from portfolio."""
    try: sharpe = pf.sharpe_ratio()
    except: sharpe = np.nan
    try: total_return = pf.total_return()
    except: total_return = np.nan
    try: max_dd = pf.max_drawdown()
    except: max_dd = np.nan
    try: n_trades = pf.trades.count()
    except: n_trades = 0
    try: win_rate = pf.trades.win_rate()
    except: win_rate = np.nan
    try: profit_factor = pf.trades.profit_factor()
    except: profit_factor = np.nan
    try:
        returns = pf.trades.returns.values
        expectancy = np.mean(returns) if len(returns) > 0 else np.nan
    except: expectancy = np.nan
    try:
        wins = returns[returns > 0]
        losses = returns[returns <= 0]
        avg_win = np.mean(wins) if len(wins) > 0 else 0
        avg_loss = abs(np.mean(losses)) if len(losses) > 0 else 1
        payoff = avg_win / avg_loss if avg_loss > 0 else np.nan
    except: payoff = np.nan
    return {
        'sharpe': sharpe, 'total_return': total_return, 'max_dd': max_dd,
        'n_trades': n_trades, 'win_rate': win_rate, 'profit_factor': profit_factor,
        'expectancy': expectancy, 'payoff': payoff
    }


# ---- RUN GRID SEARCH ----
results_list = []
combo_count = 0

for scenario in scenarios:
    for holding_days in holding_days_range:
        for require_consec in require_consec_range:
            for min_range_pct in min_range_pct_range:
                combo_count += 1
                
                try:
                    entries, exits, direction = generate_scenario_signals(
                        train_close, train_high, train_low,
                        scenario=scenario,
                        holding_days=holding_days,
                        require_consec=require_consec,
                        min_range_pct=min_range_pct
                    )
                    
                    if entries.sum() < 2:
                        continue
                    
                    pf = run_backtest(entries, exits, train_close, direction=direction)
                    m = get_metrics(pf)
                    
                    if m['n_trades'] < MIN_TRADES:
                        continue
                    
                    results_list.append({
                        'scenario': scenario,
                        'holding_days': holding_days,
                        'require_consec': require_consec,
                        'min_range_pct': min_range_pct,
                        'direction': direction,
                        **m
                    })
                except Exception as e:
                    pass
                
                if combo_count % 20 == 0:
                    print(f'Completed {combo_count}/{total_combos} combinations...')

print(f'\nCompleted {combo_count} combinations. Valid results: {len(results_list)}')

# Sort by Sharpe
results_list.sort(key=lambda x: x['sharpe'] if pd.notna(x['sharpe']) else -999, reverse=True)

# Display top 10
print(f'\nTop 10 by IS Sharpe:')
print('=' * 120)
print(f'{"Rank":<5} {"Scenario":<20} {"Hold":<5} {"Consec":<7} {"MinRng":<8} {"Sharpe":>8} {"Return":>10} {"MaxDD":>10} {"Trades":>7} {"WinRate":>8} {"PF":>8}')
print('-' * 120)
for i, r in enumerate(results_list[:10]):
    s = f"{r['sharpe']:.3f}" if pd.notna(r['sharpe']) else 'N/A'
    ret = f"{r['total_return']:.2%}" if pd.notna(r['total_return']) else 'N/A'
    dd = f"{r['max_dd']:.2%}" if pd.notna(r['max_dd']) else 'N/A'
    wr = f"{r['win_rate']:.1%}" if pd.notna(r['win_rate']) else 'N/A'
    pf = f"{r['profit_factor']:.2f}" if pd.notna(r['profit_factor']) else 'N/A'
    print(f"{i+1:<5} {r['scenario']:<20} {r['holding_days']:<5} {str(r['require_consec']):<7} {r['min_range_pct']:<8.3f} {s:>8} {ret:>10} {dd:>10} {r['n_trades']:>7} {wr:>8} {pf:>8}')

In [ ]:
# VALIDATE TOP 5 ON OOS DATA

top_5 = results_list[:5]

if len(top_5) == 0:
    print('No valid results to validate.')
else:
    print('Top 5 IS performers -- OOS Validation')
    print('=' * 130)
    print(f'{"Rank":<5} {"Scenario":<20} {"Hold":<5} {"IS Sharpe":>10} {"OOS Sharpe":>11} {"IS Return":>10} {"OOS Return":>11} {"IS MaxDD":>10} {"OOS MaxDD":>11} {"OOS Trades":>10}')
    print('-' * 130)
    
    fig, axes = plt.subplots(1, min(len(top_5), 5), figsize=(20, 5), squeeze=False)
    
    for i, r in enumerate(top_5):
        # Run on validation data
        ent_val, ext_val, direction = generate_scenario_signals(
            val_close, val_high, val_low,
            scenario=r['scenario'],
            holding_days=r['holding_days'],
            require_consec=r['require_consec'],
            min_range_pct=r['min_range_pct']
        )
        pf_val = run_backtest(ent_val, ext_val, val_close, direction=direction)
        m_val = get_metrics(pf_val)
        
        is_s = f"{r['sharpe']:.3f}" if pd.notna(r['sharpe']) else 'N/A'
        oos_s = f"{m_val['sharpe']:.3f}" if pd.notna(m_val['sharpe']) else 'N/A'
        is_r = f"{r['total_return']:.2%}" if pd.notna(r['total_return']) else 'N/A'
        oos_r = f"{m_val['total_return']:.2%}" if pd.notna(m_val['total_return']) else 'N/A'
        is_d = f"{r['max_dd']:.2%}" if pd.notna(r['max_dd']) else 'N/A'
        oos_d = f"{m_val['max_dd']:.2%}" if pd.notna(m_val['max_dd']) else 'N/A'
        
        print(f"{i+1:<5} {r['scenario']:<20} {r['holding_days']:<5} {is_s:>10} {oos_s:>11} {is_r:>10} {oos_r:>11} {is_d:>10} {oos_d:>11} {m_val['n_trades']:>10}")
        
        # Plot equity curve
        equity = pf_val.value()
        ax = axes[0][i]
        ax.plot(equity.index, equity.values, linewidth=1.5)
        ax.set_title(f'#{i+1} {r["scenario"]} h={r["holding_days"]}\nOOS Sharpe: {oos_s}', fontsize=9)
        ax.set_ylabel('Portfolio Value ($)')
        ax.tick_params(axis='x', rotation=45, labelsize=7)
    
    plt.tight_layout()
    plt.show()

## Sensitivity Analysis

Perturb each parameter from the best combo to check robustness.

In [ ]:
# SENSITIVITY ANALYSIS

if len(results_list) == 0:
    print('No results for sensitivity analysis.')
else:
    best = results_list[0]
    base_sharpe = best['sharpe']
    print(f'Base combo: scenario={best["scenario"]}, holding_days={best["holding_days"]}, '
          f'require_consec={best["require_consec"]}, min_range_pct={best["min_range_pct"]}')
    print(f'Base IS Sharpe: {base_sharpe:.3f}')
    
    # Parameters to test sensitivity on
    param_configs = [
        ('holding_days', holding_days_range),
        ('min_range_pct', min_range_pct_range),
    ]
    
    fig, axes = plt.subplots(2, len(param_configs), figsize=(12, 8))
    if len(param_configs) == 1:
        axes = axes.reshape(2, 1)
    
    sensitivity_summary = []
    
    for col, (param_name, param_values) in enumerate(param_configs):
        is_sharpes = []
        oos_sharpes = []
        
        for val in param_values:
            kwargs = {
                'scenario': best['scenario'],
                'holding_days': best['holding_days'],
                'require_consec': best['require_consec'],
                'min_range_pct': best['min_range_pct'],
            }
            kwargs[param_name] = val
            
            # IS
            ent, ext, dirn = generate_scenario_signals(train_close, train_high, train_low, **kwargs)
            if ent.sum() >= 2:
                pf_is = run_backtest(ent, ext, train_close, direction=dirn)
                is_sharpes.append(get_metrics(pf_is)['sharpe'])
            else:
                is_sharpes.append(np.nan)
            
            # OOS
            ent_v, ext_v, dirn = generate_scenario_signals(val_close, val_high, val_low, **kwargs)
            if ent_v.sum() >= 2:
                pf_oos = run_backtest(ent_v, ext_v, val_close, direction=dirn)
                oos_sharpes.append(get_metrics(pf_oos)['sharpe'])
            else:
                oos_sharpes.append(np.nan)
        
        # Compute pct change from base for coloring
        for row, (sharpes, label) in enumerate([(is_sharpes, 'IS'), (oos_sharpes, 'OOS')]):
            ax = axes[row][col]
            colors = []
            for s in sharpes:
                if pd.isna(s) or base_sharpe == 0:
                    colors.append('gray')
                else:
                    pct = (s - base_sharpe) / abs(base_sharpe) if base_sharpe != 0 else 0
                    if pct > 0.10: colors.append('#006400')
                    elif pct >= 0: colors.append('#90EE90')
                    elif pct >= -0.10: colors.append('#FFA500')
                    else: colors.append('#DC143C')
            
            x_labels = [str(v) for v in param_values]
            vals_plot = [s if pd.notna(s) else 0 for s in sharpes]
            ax.bar(x_labels, vals_plot, color=colors, alpha=0.8)
            
            # Mark base value
            base_val = best[param_name]
            if base_val in param_values:
                base_x = param_values.index(base_val)
                ax.axvline(x=base_x, color='blue', linestyle='--', alpha=0.7, linewidth=2)
            
            ax.set_title(f'{label} Sharpe vs {param_name}')
            ax.set_ylabel('Sharpe')
            ax.axhline(y=0, color='black', linewidth=0.5)
        
        # Sensitivity flag
        valid_is = [s for s in is_sharpes if pd.notna(s)]
        if len(valid_is) > 1:
            spread = (max(valid_is) - min(valid_is)) / abs(base_sharpe) if base_sharpe != 0 else 0
            flag = 'LOW' if spread < 0.30 else 'HIGH'
            sensitivity_summary.append((param_name, spread, flag))
    
    plt.tight_layout()
    plt.show()
    
    print('\nSENSITIVITY SUMMARY')
    print('=' * 50)
    for name, spread, flag in sensitivity_summary:
        icon = 'LOW' if flag == 'LOW' else 'HIGH'
        print(f'  {name:20s}  spread={spread:.1%}  [{icon}]')

In [ ]:
# FULL-SAMPLE EVALUATION + BUY & HOLD COMPARISON

if len(results_list) == 0:
    print('No results to evaluate.')
else:
    best = results_list[0]
    
    # Full-sample backtest
    ent_full, ext_full, direction = generate_scenario_signals(
        close, high, low,
        scenario=best['scenario'],
        holding_days=best['holding_days'],
        require_consec=best['require_consec'],
        min_range_pct=best['min_range_pct']
    )
    pf_full = run_backtest(ent_full, ext_full, close, direction=direction)
    m_full = get_metrics(pf_full)
    
    # Buy and hold
    pf_bh = vbt.Portfolio.from_holding(close, init_cash=INIT_CASH, fees=FEES, freq='D')
    m_bh = get_metrics(pf_bh)
    
    print(f'Best Combo: scenario={best["scenario"]}, holding={best["holding_days"]}, '
          f'consec={best["require_consec"]}, min_range={best["min_range_pct"]}')
    print(f'Direction: {direction}')
    print()
    print(f'{"Metric":<20} {"Strategy":>12} {"Buy & Hold":>12}')
    print('-' * 50)
    print(f'{"Sharpe":<20} {m_full["sharpe"]:>12.3f} {m_bh["sharpe"]:>12.3f}')
    print(f'{"Total Return":<20} {m_full["total_return"]:>12.2%} {m_bh["total_return"]:>12.2%}')
    print(f'{"Max Drawdown":<20} {m_full["max_dd"]:>12.2%} {m_bh["max_dd"]:>12.2%}')
    print(f'{"Trades":<20} {m_full["n_trades"]:>12d} {"N/A":>12}')
    print(f'{"Win Rate":<20} {m_full["win_rate"]:>12.1%} {"N/A":>12}')
    
    # Equity curve + drawdown
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), gridspec_kw={'height_ratios': [3, 1]})
    
    equity_strat = pf_full.value()
    equity_bh = pf_bh.value()
    
    ax1.plot(equity_strat.index, equity_strat.values, label=f'Scenario ({best["scenario"]})', linewidth=1.5, color='blue')
    ax1.plot(equity_bh.index, equity_bh.values, label='Buy & Hold', linewidth=1.5, color='gray', alpha=0.7)
    
    # Train/val split line
    split_date = close.index[split_idx]
    ax1.axvline(x=split_date, color='red', linestyle='--', alpha=0.7, label='Train/Val Split')
    
    # Stats box
    stats_text = (f'Sharpe: {m_full["sharpe"]:.3f}\n'
                  f'Return: {m_full["total_return"]:.2%}\n'
                  f'MaxDD: {m_full["max_dd"]:.2%}\n'
                  f'Trades: {m_full["n_trades"]}\n'
                  f'WinRate: {m_full["win_rate"]:.1%}')
    ax1.text(0.02, 0.98, stats_text, transform=ax1.transAxes, fontsize=9,
             verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    ax1.set_title(f'Scenario Strategy ({best["scenario"]}) - {TICKER} Full Sample')
    ax1.set_ylabel('Portfolio Value ($)')
    ax1.legend(loc='upper left', bbox_to_anchor=(0.15, 1.0))
    
    # Drawdown
    dd = pf_full.drawdown()
    ax2.fill_between(dd.index, dd.values, 0, color='red', alpha=0.3)
    ax2.plot(dd.index, dd.values, color='red', linewidth=0.8)
    ax2.set_title('Drawdown')
    ax2.set_ylabel('Drawdown')
    ax2.set_xlabel('Date')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# TRADE-BY-TRADE P&L ANALYSIS

if len(results_list) == 0:
    print('No results.')
else:
    best = results_list[0]
    ent_full, ext_full, direction = generate_scenario_signals(
        close, high, low,
        scenario=best['scenario'], holding_days=best['holding_days'],
        require_consec=best['require_consec'], min_range_pct=best['min_range_pct']
    )
    pf_full = run_backtest(ent_full, ext_full, close, direction=direction)
    
    try:
        trades = pf_full.trades.records_readable
        trade_returns = trades['Return'].values if 'Return' in trades.columns else trades['PnL'].values / INIT_CASH
    except:
        trade_returns = np.array([])
    
    if len(trade_returns) > 0:
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        
        # 1. Return distribution
        ax = axes[0][0]
        ax.hist(trade_returns * 100, bins=30, color='steelblue', alpha=0.7, edgecolor='black')
        ax.axvline(x=0, color='red', linewidth=1)
        ax.axvline(x=np.mean(trade_returns)*100, color='green', linewidth=2, linestyle='--', label=f'Mean: {np.mean(trade_returns)*100:.2f}%')
        ax.set_title('Trade Return Distribution')
        ax.set_xlabel('Return (%)')
        ax.legend()
        
        # 2. Cumulative P&L
        ax = axes[0][1]
        cum_returns = np.cumsum(trade_returns)
        ax.plot(cum_returns, color='blue', linewidth=1.5)
        ax.fill_between(range(len(cum_returns)), cum_returns, 0,
                        where=cum_returns >= 0, color='green', alpha=0.2)
        ax.fill_between(range(len(cum_returns)), cum_returns, 0,
                        where=cum_returns < 0, color='red', alpha=0.2)
        ax.set_title('Cumulative Trade Returns')
        ax.set_xlabel('Trade #')
        ax.set_ylabel('Cumulative Return')
        
        # 3. Win/Loss by day of week
        ax = axes[1][0]
        try:
            entry_dates = pd.to_datetime(trades['Entry Timestamp'] if 'Entry Timestamp' in trades.columns else trades['entry_date'])
            entry_dow = entry_dates.dt.dayofweek
            dow_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri']
            wins_by_day = []
            losses_by_day = []
            for d in range(5):
                mask = entry_dow == d
                day_rets = trade_returns[mask.values] if mask.sum() > 0 else np.array([])
                wins_by_day.append(np.sum(day_rets > 0))
                losses_by_day.append(np.sum(day_rets <= 0))
            x = np.arange(5)
            ax.bar(x - 0.2, wins_by_day, 0.4, color='green', alpha=0.7, label='Wins')
            ax.bar(x + 0.2, losses_by_day, 0.4, color='red', alpha=0.7, label='Losses')
            ax.set_xticks(x)
            ax.set_xticklabels(dow_labels)
            ax.set_title('Win/Loss by Entry Day')
            ax.legend()
        except:
            ax.text(0.5, 0.5, 'Could not parse entry dates', ha='center', va='center', transform=ax.transAxes)
        
        # 4. Rolling win rate (20-trade window)
        ax = axes[1][1]
        if len(trade_returns) >= 20:
            wins = (trade_returns > 0).astype(float)
            rolling_wr = pd.Series(wins).rolling(20).mean()
            ax.plot(rolling_wr.values, color='purple', linewidth=1.5)
            ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
            ax.set_title('Rolling Win Rate (20-trade window)')
            ax.set_xlabel('Trade #')
            ax.set_ylabel('Win Rate')
            ax.set_ylim(0, 1)
        else:
            ax.text(0.5, 0.5, 'Not enough trades for rolling analysis', ha='center', va='center', transform=ax.transAxes)
        
        plt.tight_layout()
        plt.show()
    else:
        print('No trades to analyze.')

In [ ]:
# MONTE CARLO FTMO SIMULATION (10,000 paths)

FTMO_ACCOUNT = 100_000
FTMO_PROFIT_TARGET = 0.10
FTMO_DAILY_LOSS_LIMIT = 0.05
FTMO_TOTAL_LOSS_LIMIT = 0.10
FTMO_DAYS = 30
N_SIMULATIONS = 10_000

if len(results_list) == 0:
    print('No results.')
else:
    best = results_list[0]
    ent_full, ext_full, direction = generate_scenario_signals(
        close, high, low,
        scenario=best['scenario'], holding_days=best['holding_days'],
        require_consec=best['require_consec'], min_range_pct=best['min_range_pct']
    )
    pf_full = run_backtest(ent_full, ext_full, close, direction=direction)
    daily_returns = pf_full.daily_returns().dropna().values
    
    if len(daily_returns) < 10:
        print('Not enough daily returns for Monte Carlo.')
    else:
        pass_count = 0
        fail_daily = 0
        fail_total = 0
        fail_no_target = 0
        final_balances = []
        
        for sim in range(N_SIMULATIONS):
            # Sample FTMO_DAYS returns with replacement
            sampled = np.random.choice(daily_returns, size=FTMO_DAYS, replace=True)
            
            balance = FTMO_ACCOUNT
            peak = FTMO_ACCOUNT
            passed = False
            failed = False
            fail_reason = ''
            
            for day_ret in sampled:
                daily_pnl = balance * day_ret
                balance += daily_pnl
                
                # Check daily loss limit
                if daily_pnl < -FTMO_ACCOUNT * FTMO_DAILY_LOSS_LIMIT:
                    failed = True
                    fail_reason = 'daily'
                    break
                
                # Check total loss limit
                if balance < FTMO_ACCOUNT * (1 - FTMO_TOTAL_LOSS_LIMIT):
                    failed = True
                    fail_reason = 'total'
                    break
                
                # Check profit target
                if balance >= FTMO_ACCOUNT * (1 + FTMO_PROFIT_TARGET):
                    passed = True
                    break
            
            final_balances.append(balance)
            if passed:
                pass_count += 1
            elif failed and fail_reason == 'daily':
                fail_daily += 1
            elif failed and fail_reason == 'total':
                fail_total += 1
            else:
                fail_no_target += 1
        
        pass_rate = pass_count / N_SIMULATIONS
        
        print(f'FTMO Monte Carlo Simulation ({N_SIMULATIONS:,} paths, {FTMO_DAYS} days each)')
        print('=' * 60)
        print(f'Pass Rate:           {pass_rate:.1%}')
        print(f'Failed (Daily Loss): {fail_daily/N_SIMULATIONS:.1%}')
        print(f'Failed (Total Loss): {fail_total/N_SIMULATIONS:.1%}')
        print(f'Failed (No Target):  {fail_no_target/N_SIMULATIONS:.1%}')
        print(f'\nMedian Final Balance: ${np.median(final_balances):,.0f}')
        print(f'Mean Final Balance:   ${np.mean(final_balances):,.0f}')
        
        verdict = 'PASS' if pass_rate >= 0.50 else 'MARGINAL' if pass_rate >= 0.30 else 'FAIL'
        print(f'\nVerdict: {verdict}')
        
        # Plot
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
        
        ax1.hist(final_balances, bins=50, color='steelblue', alpha=0.7, edgecolor='black')
        ax1.axvline(x=FTMO_ACCOUNT, color='gray', linestyle='--', label='Starting Balance')
        ax1.axvline(x=FTMO_ACCOUNT*(1+FTMO_PROFIT_TARGET), color='green', linestyle='--', label='Profit Target')
        ax1.axvline(x=FTMO_ACCOUNT*(1-FTMO_TOTAL_LOSS_LIMIT), color='red', linestyle='--', label='Loss Limit')
        ax1.set_title(f'Final Balance Distribution (Pass Rate: {pass_rate:.1%})')
        ax1.set_xlabel('Final Balance ($)')
        ax1.legend(fontsize=8)
        
        # Pie chart
        labels = ['Passed', 'Failed (Daily)', 'Failed (Total)', 'No Target']
        sizes = [pass_count, fail_daily, fail_total, fail_no_target]
        colors_pie = ['#2ecc71', '#e74c3c', '#c0392b', '#f39c12']
        non_zero = [(l, s, c) for l, s, c in zip(labels, sizes, colors_pie) if s > 0]
        if non_zero:
            ax2.pie([x[1] for x in non_zero], labels=[x[0] for x in non_zero],
                    colors=[x[2] for x in non_zero], autopct='%1.1f%%', startangle=90)
        ax2.set_title('FTMO Outcome Distribution')
        
        plt.tight_layout()
        plt.show()

In [ ]:
# UNIVERSAL EXPORT + GOOGLE SHEETS LOGGING

if len(results_list) == 0:
    print('No results to export.')
else:
    best = results_list[0]
    
    # Full sample backtest for export
    ent_full, ext_full, direction = generate_scenario_signals(
        close, high, low,
        scenario=best['scenario'], holding_days=best['holding_days'],
        require_consec=best['require_consec'], min_range_pct=best['min_range_pct']
    )
    pf_full = run_backtest(ent_full, ext_full, close, direction=direction)
    m_full = get_metrics(pf_full)
    
    # IS metrics
    ent_is, ext_is, dir_is = generate_scenario_signals(
        train_close, train_high, train_low,
        scenario=best['scenario'], holding_days=best['holding_days'],
        require_consec=best['require_consec'], min_range_pct=best['min_range_pct']
    )
    pf_is = run_backtest(ent_is, ext_is, train_close, direction=dir_is)
    m_is = get_metrics(pf_is)
    
    # OOS metrics
    ent_oos, ext_oos, dir_oos = generate_scenario_signals(
        val_close, val_high, val_low,
        scenario=best['scenario'], holding_days=best['holding_days'],
        require_consec=best['require_consec'], min_range_pct=best['min_range_pct']
    )
    pf_oos = run_backtest(ent_oos, ext_oos, val_close, direction=dir_oos)
    m_oos = get_metrics(pf_oos)
    
    # Buy & hold
    pf_bh = vbt.Portfolio.from_holding(close, init_cash=INIT_CASH, fees=FEES, freq='D')
    m_bh = get_metrics(pf_bh)
    
    # MC results
    mc_pass_rate = pass_rate if 'pass_rate' in dir() else 0.0
    mc_verdict = verdict if 'verdict' in dir() else 'N/A'
    
    import uuid
    run_id = str(uuid.uuid4())[:8]
    
    summary = {
        'metadata': {
            'run_id': run_id,
            'ticker': TICKER,
            'strategy_name': f'Scenario_{best["scenario"]}',
            'strategy_family': 'Scenario_Strategy',
            'start_date': str(close.index[0].date()),
            'end_date': str(close.index[-1].date()),
            'total_bars': len(close),
            'notes': f'Hougaard Scenario Strategy. hold={best["holding_days"]}, consec={best["require_consec"]}, minrng={best["min_range_pct"]}'
        },
        'best_params': {
            'scenario': best['scenario'],
            'holding_days': best['holding_days'],
            'require_consec': best['require_consec']
        },
        'metrics_in_sample': {
            'sharpe': float(m_is['sharpe']) if pd.notna(m_is['sharpe']) else 0,
            'return': float(m_is['total_return']) if pd.notna(m_is['total_return']) else 0,
            'dd': float(m_is['max_dd']) if pd.notna(m_is['max_dd']) else 0,
            'trades': int(m_is['n_trades'])
        },
        'metrics_out_of_sample': {
            'sharpe': float(m_oos['sharpe']) if pd.notna(m_oos['sharpe']) else 0,
            'return': float(m_oos['total_return']) if pd.notna(m_oos['total_return']) else 0,
            'dd': float(m_oos['max_dd']) if pd.notna(m_oos['max_dd']) else 0,
            'trades': int(m_oos['n_trades'])
        },
        'metrics_full_sample': {
            'sharpe': float(m_full['sharpe']) if pd.notna(m_full['sharpe']) else 0,
            'total_return': float(m_full['total_return']) if pd.notna(m_full['total_return']) else 0,
            'max_dd': float(m_full['max_dd']) if pd.notna(m_full['max_dd']) else 0,
            'trades': int(m_full['n_trades']),
            'win_rate': float(m_full['win_rate']) if pd.notna(m_full['win_rate']) else 0,
            'pf': float(m_full['profit_factor']) if pd.notna(m_full['profit_factor']) else 0,
            'expectancy': float(m_full['expectancy']) if pd.notna(m_full['expectancy']) else 0,
            'payoff': float(m_full['payoff']) if pd.notna(m_full['payoff']) else 0
        },
        'metrics_buy_hold': {
            'sharpe': float(m_bh['sharpe']) if pd.notna(m_bh['sharpe']) else 0,
            'return': float(m_bh['total_return']) if pd.notna(m_bh['total_return']) else 0
        },
        'monte_carlo_ftmo': {
            'pass_rate': mc_pass_rate,
            'verdict': mc_verdict
        }
    }
    
    # Save JSON
    import json
    strategy_dir = f'Scenario_{best["scenario"]}'
    
    # Try Google Drive first (Colab), fall back to local
    try:
        base_dir = f'/content/drive/MyDrive/strategy_exports/{strategy_dir}/{TICKER}/latest'
        os.makedirs(base_dir, exist_ok=True)
    except:
        base_dir = os.path.join(os.path.dirname(os.path.abspath('.')), 'exports', strategy_dir, TICKER, 'latest')
        os.makedirs(base_dir, exist_ok=True)
    
    json_path = os.path.join(base_dir, 'summary.json')
    with open(json_path, 'w') as f:
        json.dump(summary, f, indent=2)
    print(f'Summary saved to: {json_path}')
    
    # Save trades CSV
    try:
        trades_df = pf_full.trades.records_readable
        trades_path = os.path.join(base_dir, 'trades.csv')
        trades_df.to_csv(trades_path, index=False)
        print(f'Trades saved to: {trades_path}')
    except Exception as e:
        print(f'Could not save trades: {e}')
    
    # Save daily returns CSV
    daily_ret_series = pf_full.daily_returns()
    ret_path = os.path.join(base_dir, 'daily_returns.csv')
    daily_ret_series.to_csv(ret_path)
    print(f'Daily returns saved to: {ret_path}')
    
    # Save grid results CSV
    grid_df = pd.DataFrame(results_list)
    grid_path = os.path.join(base_dir, 'grid_results.csv')
    grid_df.to_csv(grid_path, index=False)
    print(f'Grid results saved to: {grid_path}')
    
    # Google Sheets logging
    try:
        from lib.sheets_logger import SheetsLogger
        logger = SheetsLogger()
        logger.log_result(summary)
        
        # Also log trades
        try:
            trades_df = pf_full.trades.records_readable
            logger.log_trades(TICKER, f'Scenario_{best["scenario"]}', run_id, trades_df)
        except:
            pass
    except Exception as e:
        print(f'Google Sheets logging skipped: {e}')
    
    print('\nExport complete!')